# 04 — Probability Calibration

This notebook reproduces the public-sample calibration workflow. It compares model fair probabilities and market-implied probabilities against realized settlement outcomes.

Calibration is not a profitability test. It asks whether forecast probabilities behave like probabilities.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "sample"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

pd.set_option("display.max_columns", 80)


In [ ]:
from models.calibration import load_forecast_outcomes, summarize_calibration

forecasts = load_forecast_outcomes(
    DATA_DIR / "tick_snapshots_sample.csv",
    DATA_DIR / "settlements_sample.csv",
)
fair_summary = summarize_calibration(forecasts, source="fair")
market_summary = summarize_calibration(forecasts, source="market")

print(f"joined market-level observations: {len(forecasts)}")
print(f"fair brier score: {fair_summary.brier_score:.4f}")
print(f"market brier score: {market_summary.brier_score:.4f}")


In [ ]:
def summary_table(summary):
    return {
        "source": summary.label,
        "observations": summary.observations,
        "brier_score": summary.brier_score,
        "log_loss": summary.log_loss,
    }

pd.DataFrame([summary_table(fair_summary), summary_table(market_summary)])


## Bucket diagnostics

The bucket table shows average forecast probability, realized outcome rate, and absolute calibration error by probability bucket.


In [ ]:
def buckets_to_frame(summary):
    return pd.DataFrame(
        [
            {
                "source": summary.label,
                "bucket": bucket.bucket,
                "count": bucket.count,
                "avg_forecast": bucket.avg_forecast,
                "realized_rate": bucket.realized_rate,
                "avg_abs_error": bucket.avg_abs_error,
            }
            for bucket in summary.calibration_buckets
        ]
    )

bucket_frame = pd.concat(
    [buckets_to_frame(fair_summary), buckets_to_frame(market_summary)],
    ignore_index=True,
)
bucket_frame


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for source, group in bucket_frame.groupby("source"):
    ax.plot(group["avg_forecast"], group["realized_rate"], marker="o", label=source)
ax.plot([0, 1], [0, 1], linestyle="--", label="perfect calibration")
ax.set_title("Calibration by probability bucket")
ax.set_xlabel("Average forecast")
ax.set_ylabel("Realized rate")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## Extreme probability buckets

The public sample suggests that the middle probability range is more stable than the extreme buckets. Extreme buckets have fewer observations, so they naturally have higher variance. They are also where final-resolution-window behavior can be hardest to model.


In [ ]:
extreme_view = bucket_frame[
    bucket_frame["bucket"].isin(["0.0-0.1", "0.1-0.2", "0.8-0.9", "0.9-1.0"])
].sort_values(["source", "bucket"])
extreme_view


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_frame = bucket_frame.copy()
plot_frame["bucket_source"] = plot_frame["source"] + " " + plot_frame["bucket"]
ax.bar(plot_frame["bucket_source"], plot_frame["avg_abs_error"])
ax.set_title("Calibration absolute error by bucket")
ax.set_ylabel("Avg absolute error")
ax.tick_params(axis="x", rotation=75)
ax.grid(True, axis="y", alpha=0.3)
plt.show()


## Author takeaway

The calibration pattern is most interesting in the tails. My interpretation is that extreme probabilities in a five-minute binary market often occur either after a large early reference-price move or inside the final resolution window. In the last 5-10 seconds, price can reverse or repeatedly cross the opening anchor while probabilities are already compressed near 0 or 1. This can create large forecast error without requiring a large BTC move.

This remains a public-sample diagnostic, not a complete lead-lag or oracle-reference validation study.
